# 02b — GRU Discovery Tuning

Standalone discovery notebook (peer of `02_tune_nn.ipynb`) that tests whether a
compact symmetric GRU over each player's ten causal prior matches can match the
tabular MLP's log loss. It reuses the exact chronological train/validation/test
split and the match-grouped time-forward CV assignments, but writes only a local
`gru_discovery_results.json`. It registers, promotes, or packages nothing, and it
does not depend on `NB_ORDER` or any injected Papermill parameter.


In [ ]:
import os
from src.constants import (
    CV_FOLDS,
    DATA_PROCESSED,
    MODELS_ARTIFACTS,
    OUTPUTS,
    load_env,
    RECENCY_HALF_LIFE_DAYS,
    RECENCY_HALF_LIFE_KEY,
    RECENCY_CUTOFF_KEY,
)
from src.db.snapshot import SNAPSHOT_PATH
import numpy as np
import pandas as pd

random_state = 42
load_env()

input_dir = str(DATA_PROCESSED)

# Results stay discovery-local; the full history export is the only shared
# model-directory artifact written by this notebook.
discovery_dir = OUTPUTS / "gru_discovery"
discovery_dir.mkdir(parents=True, exist_ok=True)
output_path = discovery_dir / "gru_discovery_results.json"

# Fixed parameters for the no-Optuna run.
fixed_params = {
    "lr": 0.001,
    "weight_decay": 0.00001,
    "dropout": 0.2,
}
fixed_epochs = 3

# Early stopping / epoch caps mirror the tabular NN notebook.
max_epochs = 50
patience = 5
batch_size = 4096
eval_batch_size = batch_size * 2
num_workers = 4
print("CELL 1/6: configuration ready", flush=True)

In [ ]:
import io
import json
import logging
import os
import re
import warnings
import time
from contextlib import redirect_stdout, redirect_stderr

os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
os.environ["DO_NOT_TRACK"] = "true"
logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.tips").disabled = True
logging.getLogger("optuna").setLevel(logging.INFO)
logging.disable(logging.INFO)
warnings.filterwarnings("ignore", message=r"Checkpoint directory .* exists and is not empty\.")
warnings.filterwarnings("ignore", message=r".*treespec.*deprecated.*")
warnings.filterwarnings("ignore", message=r".*leaked semaphore.*")

import torch
import torch.nn as nn
import lightning as L
import optuna
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from optuna_integration.pytorch_lightning import PyTorchLightningPruningCallback
from src.training.grouped_cv import (
    grouped_fold_indices,
    assert_groups_intact,
    load_validated_fold_assignment,
)
from src.training.nn import SymmetricGRU
from src.training.gru_history import (
    STORE_WIDTH,
    HISTORY_LEN,
    read_player_match_history_df,
    build_history_store,
    map_split_indices,
    build_context_tensor,
)

print("CELL 2/6: dependencies imported", flush=True)

In [ ]:
# ── Load the existing split frames (same as the tabular NN notebook) ──
print("CELL 3/6: loading split parquet files", flush=True)
X_train = pd.read_parquet(f"{input_dir}/X_train.parquet")
X_val = pd.read_parquet(f"{input_dir}/X_val.parquet")
X_test = pd.read_parquet(f"{input_dir}/X_test.parquet")
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"]
y_val = pd.read_parquet(f"{input_dir}/y_val.parquet")["y"]
y_test = pd.read_parquet(f"{input_dir}/y_test.parquet")["y"]
info_train = pd.read_parquet(f"{input_dir}/info_train.parquet")
info_val = pd.read_parquet(f"{input_dir}/info_val.parquet")
info_test = pd.read_parquet(f"{input_dir}/info_test.parquet")
train_cutoff = max(info_train["match_date"].max(), info_test["match_date"].max())

# Shared match-level fold map written by the 01 split notebook; validated, never
# written here.
fold_frame = load_validated_fold_assignment(
    info_train["match_id"],
    info_train["match_date"],
    CV_FOLDS,
    random_state,
    f"{input_dir}/fold_assignment.parquet",
)

print(f"Using full dataset: {len(X_train):,}/{len(X_val):,}/{len(X_test):,} rows", flush=True)

# ── Build the causal history store once, shared by every trial ──
prep_start = time.time()
print("CELL 3/6: reading all player matches from DuckDB snapshot", flush=True)
history_df = read_player_match_history_df(SNAPSHOT_PATH)
print(f"Loaded {len(history_df):,} player-match rows; building causal histories", flush=True)
store = build_history_store(history_df)
print("Causal history store built; exporting full store to parquet", flush=True)
history_columns = {
    "player_id": store.player_ids,
    "match_id": store.match_ids,
}
for step in range(HISTORY_LEN):
    history_columns[f"valid_{step}"] = store.valid_mask[:, step]
    for feature_index in range(STORE_WIDTH):
        history_columns[f"h{step}_{feature_index}"] = store.histories[:, step, feature_index]
history_export = pd.DataFrame(history_columns)
history_export_path = MODELS_ARTIFACTS / "gru_history.parquet"
history_export.to_parquet(history_export_path, index=False)
print(f"Wrote full history parquet: {history_export_path}", flush=True)

# Map each directional split row to player/opponent history-store indices once.
train_p_idx, train_o_idx = map_split_indices(store, info_train)
val_p_idx, val_o_idx = map_split_indices(store, info_val)
test_p_idx, test_o_idx = map_split_indices(store, info_test)

# Reduced current context, rebuilt from the tabular X_* frames.
ctx_train = build_context_tensor(X_train)
ctx_val = build_context_tensor(X_val)
ctx_test = build_context_tensor(X_test)

times = {}
times["preprocessing_seconds"] = time.time() - prep_start

print(
    f"History store: {store.histories.shape} (N x {HISTORY_LEN} x {STORE_WIDTH}); "
    f"context dim {ctx_train.shape[1]}"
)
print(f"Train/val/test rows: {len(X_train)}/{len(X_val)}/{len(X_test)}")
print(f"Preprocessing time: {times['preprocessing_seconds']:.1f}s", flush=True)
del history_df, history_export, history_columns
import gc

gc.collect()
print(f"Using {num_workers} DataLoader workers", flush=True)
print("CELL 3/6: preprocessing complete", flush=True)

In [ ]:
# ── Per-fit-band normalization helpers ──
# Imputation fill and context scaling are fit ONLY on the fit rows of each band,
# so OOF folds and the validation/refit bands never see later or test statistics.


def fit_band_norm(fit_store_idx, ctx_fit):
    imputed = store.impute(np.unique(np.asarray(fit_store_idx, dtype=np.int64)))
    scaler = StandardScaler().fit(np.asarray(ctx_fit, dtype=np.float32))
    return imputed, scaler


def gather_tensors(imputed, scaler, p_idx, o_idx, ctx):
    ph, oh, pv, ov = store.gather(imputed, np.asarray(p_idx), np.asarray(o_idx))
    ctx_s = scaler.transform(np.asarray(ctx, dtype=np.float32)).astype(np.float32)
    return (
        torch.from_numpy(ph),
        torch.from_numpy(oh),
        torch.from_numpy(pv),
        torch.from_numpy(ov),
        torch.from_numpy(ctx_s),
    )


def make_ds(tensors5, y):
    return TensorDataset(*tensors5, torch.from_numpy(np.asarray(y, dtype=np.float32)))


checkpoint_dir = discovery_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)


def make_trainer(trial=None):
    checkpoint = ModelCheckpoint(
        dirpath=str(checkpoint_dir / (f"trial_{trial.number}" if trial else "final")),
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=patience, mode="min"),
        checkpoint,
    ]
    if trial is not None:
        callbacks.append(PyTorchLightningPruningCallback(trial, monitor="val_loss"))
    return (
        L.Trainer(
            max_epochs=max_epochs,
            accelerator="auto",
            default_root_dir=str(checkpoint_dir),
            enable_progress_bar=False,
            enable_model_summary=False,
            callbacks=callbacks,
            logger=False,
        ),
        checkpoint,
    )


def fit_predict_loss(model, trial, fit_ds, es_ds, val_ds):
    """Fit on fit_ds (early stopping on es_ds), return predictions, model, checkpoint, and loss."""
    trainer, checkpoint = make_trainer(trial)
    trainer.fit(
        model,
        train_dataloaders=DataLoader(
            fit_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            persistent_workers=num_workers > 0,
        ),
        val_dataloaders=DataLoader(
            es_ds,
            batch_size=eval_batch_size,
            shuffle=False,
            num_workers=num_workers,
            persistent_workers=num_workers > 0,
        ),
    )
    if checkpoint.best_model_path:
        model = SymmetricGRU.load_from_checkpoint(checkpoint.best_model_path)
    preds = trainer.predict(
        model,
        dataloaders=DataLoader(
            val_ds,
            batch_size=eval_batch_size,
            shuffle=False,
            num_workers=num_workers,
            persistent_workers=num_workers > 0,
        ),
    )
    assert preds is not None
    probs = np.concatenate([np.asarray(p) for p in preds])
    labels = val_ds.tensors[-1]
    loss = nn.functional.binary_cross_entropy(torch.from_numpy(probs), labels).item()
    return probs, model, checkpoint, loss


def build_model(params):
    return SymmetricGRU(
        hist_dim=STORE_WIDTH,
        context_dim=ctx_train.shape[1],
        hidden_dim=32,  # fixed
        dropout=params["dropout"],
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )


# ── Optuna objective: fit train band, early-stop on val band, score val band ──
# Test labels are never touched by selection, pruning, or early stopping.
def objective(trial):
    params = {
        "lr": trial.suggest_float("lr", 3e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True),
        "dropout": trial.suggest_float("dropout", 0.0, 0.2),
    }
    print(f"Trial {trial.number} starting: params={params}", flush=True)
    imputed, scaler = fit_band_norm(np.concatenate([train_p_idx, train_o_idx]), ctx_train)
    train_t5 = gather_tensors(imputed, scaler, train_p_idx, train_o_idx, ctx_train)
    val_t5 = gather_tensors(imputed, scaler, val_p_idx, val_o_idx, ctx_val)
    probs, _, checkpoint, loss = fit_predict_loss(
        build_model(params),
        trial,
        make_ds(train_t5, y_train),
        make_ds(val_t5, y_val),
        make_ds(val_t5, y_val),
    )
    val_auc = roc_auc_score(y_val, probs)
    best_epoch = None
    if checkpoint.best_model_path:
        match = re.search(r"epoch=(\d+)", checkpoint.best_model_path)
        best_epoch = int(match.group(1)) + 1 if match else None
    trial.set_user_attr("val_roc_auc", float(val_auc))
    trial.set_user_attr("best_epoch", best_epoch)
    if trial.should_prune():
        raise optuna.TrialPruned()
    return float(loss)


best_params = dict(fixed_params)
selection_best_epochs = fixed_epochs
times["tuning_seconds"] = 0.0
print(f"Using fixed GRU parameters: {best_params}, epochs={fixed_epochs}", flush=True)
print("CELL 4/6: fixed configuration ready; Optuna skipped", flush=True)

In [ ]:
import re

best_params = dict(fixed_params)

# ── Selection model: train band fit, early stopping on the validation band ──
sel_imputed, sel_scaler = fit_band_norm(np.concatenate([train_p_idx, train_o_idx]), ctx_train)
sel_train_t5 = gather_tensors(sel_imputed, sel_scaler, train_p_idx, train_o_idx, ctx_train)
sel_val_t5 = gather_tensors(sel_imputed, sel_scaler, val_p_idx, val_o_idx, ctx_val)

_, selection_model, selection_checkpoint, _ = fit_predict_loss(
    build_model(best_params),
    None,
    make_ds(sel_train_t5, y_train),
    make_ds(sel_val_t5, y_val),
    make_ds(sel_val_t5, y_val),
)
assert selection_checkpoint.best_model_path, "selection checkpoint missing"
_epoch_match = re.search(r"epoch=(\d+)", selection_checkpoint.best_model_path)
assert _epoch_match, f"no epoch in {selection_checkpoint.best_model_path}"
selection_best_epochs = fixed_epochs

with torch.no_grad():
    sel_val_probs = torch.sigmoid(selection_model(*sel_val_t5)).numpy()
selection_val_log_loss = log_loss(y_val, sel_val_probs)
selection_val_auc = roc_auc_score(y_val, sel_val_probs)

# ── Grouped time-forward OOF over the train split ──
print("CELL 5/6: starting selection and grouped OOF", flush=True)
oof_start = time.time()
rng = np.random.default_rng(random_state)
oof = np.full(len(X_train), np.nan)
for train_idx, val_idx in grouped_fold_indices(
    info_train["match_id"], info_train["match_date"], CV_FOLDS, random_state, y_train
):
    assert_groups_intact(info_train["match_id"], train_idx, val_idx)
    print(f"OOF fold: train={len(train_idx):,} val={len(val_idx):,}", flush=True)
    perm = rng.permutation(len(train_idx))
    n_val = max(1, len(train_idx) // 5)
    fit_idx = train_idx[perm[n_val:]]
    es_idx = train_idx[perm[:n_val]]
    imputed, scaler = fit_band_norm(
        np.concatenate([train_p_idx[fit_idx], train_o_idx[fit_idx]]),
        ctx_train[fit_idx],
    )
    fit_t5 = gather_tensors(
        imputed, scaler, train_p_idx[fit_idx], train_o_idx[fit_idx], ctx_train[fit_idx]
    )
    es_t5 = gather_tensors(
        imputed, scaler, train_p_idx[es_idx], train_o_idx[es_idx], ctx_train[es_idx]
    )
    val_t5 = gather_tensors(
        imputed, scaler, train_p_idx[val_idx], train_o_idx[val_idx], ctx_train[val_idx]
    )
    oof[val_idx], _, _, _ = fit_predict_loss(
        build_model(best_params),
        None,
        make_ds(fit_t5, y_train.iloc[fit_idx]),
        make_ds(es_t5, y_train.iloc[es_idx]),
        make_ds(val_t5, y_train.iloc[val_idx]),
    )
oof_mask = np.isfinite(oof)
assert oof_mask.any(), "no OOF predictions were generated"
oof_log_loss = log_loss(y_train[oof_mask], oof[oof_mask])
times["oof_seconds"] = time.time() - oof_start

# ── Fixed-epoch full refit on train+val, score the untouched test band ──
refit_start = time.time()
refit_store_idx = np.unique(np.concatenate([train_p_idx, train_o_idx, val_p_idx, val_o_idx]))
refit_imputed, refit_scaler = fit_band_norm(
    refit_store_idx, np.concatenate([ctx_train, ctx_val], axis=0)
)
tv_p_idx = np.concatenate([train_p_idx, val_p_idx])
tv_o_idx = np.concatenate([train_o_idx, val_o_idx])
tv_ctx = np.concatenate([ctx_train, ctx_val], axis=0)
tv_y = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
trainval_t5 = gather_tensors(refit_imputed, refit_scaler, tv_p_idx, tv_o_idx, tv_ctx)
test_t5 = gather_tensors(refit_imputed, refit_scaler, test_p_idx, test_o_idx, ctx_test)

refit_trainer = L.Trainer(
    max_epochs=selection_best_epochs,
    accelerator="auto",
    default_root_dir=str(checkpoint_dir),
    enable_progress_bar=False,
    enable_model_summary=False,
    logger=False,
    callbacks=[],  # no validation callbacks in the fixed-epoch refit
)
refit_model = build_model(best_params)
print("Refitting on train+validation and scoring untouched test", flush=True)
refit_trainer.fit(
    refit_model,
    train_dataloaders=DataLoader(
        make_ds(trainval_t5, tv_y),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        persistent_workers=num_workers > 0,
    ),
)
test_preds = refit_trainer.predict(
    refit_model,
    dataloaders=DataLoader(
        make_ds(test_t5, y_test),
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=num_workers,
        persistent_workers=num_workers > 0,
    ),
)
assert test_preds is not None
test_probs = np.concatenate([np.asarray(p) for p in test_preds])
test_log_loss = log_loss(y_test, test_probs)
times["refit_test_seconds"] = time.time() - refit_start

print(f"Selection best epochs: {selection_best_epochs}")
print(f"GRU validation log loss: {selection_val_log_loss:.4f}")
print(f"GRU OOF log loss: {oof_log_loss:.4f}")
print(f"GRU test log loss: {test_log_loss:.4f}", flush=True)
print("CELL 5/6: evaluation complete", flush=True)

In [ ]:
# ── Assemble discovery results ──
gru_metrics = {
    "validation_log_loss": float(selection_val_log_loss),
    "oof_log_loss": oof_log_loss,
    "test_log_loss": float(test_log_loss),
    "validation_roc_auc": float(selection_val_auc),
    "selection_best_epochs": int(selection_best_epochs),
}

results = {
    "model": "gru_discovery",
    "fixed_config": {
        "hidden_dim": 32,
        "n_gru_layers": 1,
        "history_len": int(HISTORY_LEN),
        "store_width": int(STORE_WIDTH),
        "context_dim": int(ctx_train.shape[1]),
        "n_trials": 0,
        "cv_folds": CV_FOLDS,
        "max_epochs": max_epochs,
        "patience": patience,
        "batch_size": batch_size,
        "random_state": random_state,
        "search_space": {
            "lr": [3e-4, 3e-3],
            "weight_decay": [1e-6, 1e-4],
            "dropout": [0.0, 0.2],
        },
    },
    "best_params": {**best_params, "hidden_dim": 32},
    "metrics": gru_metrics,
    "tensor_shapes": {
        "player_hist": list(sel_train_t5[0].shape),
        "opponent_hist": list(sel_train_t5[1].shape),
        "context": list(sel_train_t5[4].shape),
    },
    "elapsed_seconds": {
        "preprocessing": float(times["preprocessing_seconds"]),
        "tuning": float(times["tuning_seconds"]),
        "oof": float(times["oof_seconds"]),
        "refit_test": float(times["refit_test_seconds"]),
    },
}

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Wrote discovery results: {output_path}")

print("GRU metrics:", json.dumps(gru_metrics, indent=2))